# Import libraries and initialize the llm

In [21]:
import os
from operator import itemgetter
from dotenv import load_dotenv
from langchain_core.globals import set_debug
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import trim_messages
from langchain_classic.schema import Document
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
load_dotenv()

True

In [2]:
api_key = os.getenv("API_KEY")
llm = model = GoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_tokens=1500,
    timeout=None,
    max_retries=2
)

# Add memory to the indication

Based on adding the conversation history to each prompt. You can add either the whole conversation or a summary of the conversation (summarized with an LLM). You can also either use the context window or combine the previous strategies, adding both a summary and the last messages.

Function shared by the following examples

In [5]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

## Save the whole conversation history

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Your name is {name}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm

store = {}

conversational_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

conversational_chain

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x00000186DC1F8720>, input_messages_key='input', history_messages_key='history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [10]:
# Invoke the chain, passing a session ID to track the user
response1 = conversational_chain.invoke(
    {
        "name":"AI Bot",
        "input": "Hi, my name is Alex."
    },
    config={"configurable": {"session_id": "session_123"}}
)
print(f"Response 1: {response1}")

# Using the same session_id
response2 = conversational_chain.invoke(
    {
        "name":"AI Bot",
        "input": "What is my name? What is your name?"
    },
    config={"configurable": {"session_id": "session_123"}}
)
print(f"Response 2: {response2}")

# Using a different session_id
response3 = conversational_chain.invoke(
    {
        "name":"Arturito",
        "input": "What is my name? What is your name?"
    },
    config={"configurable": {"session_id": "session_124"}}
)
print(f"Response 3: {response3}")

Response 1: AI: Hi Alex, it's nice to meet you! I'm AI Bot. How can I help you today?
Response 2: AI: Your name is Alex, and my name is AI Bot.
Response 3: I do not have access to your personal information, so I cannot tell you your name. My name is Arturito.


In [11]:
store

{'session_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi, my name is Alex.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hi Alex, it's nice to meet you! I'm AI Bot. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Hi, my name is Alex.', additional_kwargs={}, response_metadata={}), AIMessage(content="AI: Hi Alex, it's nice to meet you! I'm AI Bot. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name? What is your name?', additional_kwargs={}, response_metadata={}), AIMessage(content='AI: Your name is Alex, and my name is AI Bot.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]),
 'session_124': InMemoryChatMessageHistory(messages=[HumanMessage(content='What is my name? What is your name?', additional_kwargs={}, response_metadata={}), 

# Save a summary of the conversation history

## Use create_stuff_documents_chain function (basic solution)

In [5]:
# Define a prompt that MUST include the "{context}" variable
prompt = ChatPromptTemplate.from_messages([
    (
        "system", 
        "Answer the user's question using only the provided context.\n\nContext:\n{context}"
    ),
    ("human", "{question}")
])

# 3. Create the stuff documents chain
# This returns a Runnable that expects a list of Documents in the inputs
stuff_chain = create_stuff_documents_chain(llm, prompt)

# 4. Create some mock LangChain Document objects
docs = [
    Document(page_content="El asado es una técnica de cocción tradicional en Argentina."),
    Document(page_content="Se acostumbra cocinar carne DE PESCADO a la parrilla con leña o carbón.")
]

# 5. Invoke the chain
# You pass the list of documents and any other variables your prompt needs (like 'question')
resultado = stuff_chain.invoke({
    "context": docs,
    "question": "¿Qué tipo de carne se suele utilizar para el asado?"
})

print(resultado)

carne DE PESCADO


## Use the trimmer function

### Count tokens

In [22]:
set_debug(True)

# Create the Trimmer (This replaces the "Window")
# Instead of counting interactions, the modern standard is to count tokens.
# It ensures you never exceed the context limit, regardless of message length.
trimmer = trim_messages(
    max_tokens=50,          # The size of your "window"
    strategy="last",          # Keep the most recent messages
    token_counter=llm,        # Use the LLM's specific tokenizer
    include_system=True,      # Never delete the system prompt
    allow_partial=False       # Don't chop sentences in half
)

# Define the Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Build the Chain using LCEL
# The history is fetched, piped through the trimmer, and then into the prompt
chain = (
    RunnablePassthrough.assign(
        history=itemgetter("history") | trimmer
    )
    | prompt 
    | llm
)

# Set up external memory management
store = {}

# Wrap the chain to handle the history automatically
conversational_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

In [23]:
# Interaction 1
resultado = conversational_chain.invoke(
    {"input": "Hello, I have a problem with my washing machine."},
    config={"configurable": {"session_id": "user_789"}}
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "Hello, I have a problem with my washing machine."
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "Hello, I have a problem with my washing machine."
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<history>] Entering Chain run with input:
{
  "input": "Hello, I have a problem with my washing machine."
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<history> > chain:load_history] Entering Chain run with input:
{
  "input": "Hello, I have a problem with my washing machine."
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<history> > chain:load_history] s] Exiting Chain run with output:
{
  "output": []
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:Run

In [25]:
# Interaction 2: it should remember the color
resultado = conversational_chain.invoke(
    {"input": "It displays the error code 33"},
    config={"configurable": {"session_id": "user_789"}}
)

[chain/start] [chain:RunnableWithMessageHistory] Entering Chain run with input:
{
  "input": "It displays the error code 33"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history] Entering Chain run with input:
{
  "input": "It displays the error code 33"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<history>] Entering Chain run with input:
{
  "input": "It displays the error code 33"
}
[chain/start] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<history> > chain:load_history] Entering Chain run with input:
{
  "input": "It displays the error code 33"
}
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<history> > chain:load_history] s] Exiting Chain run with output:
[outputs]
[chain/end] [chain:RunnableWithMessageHistory > chain:insert_history > chain:RunnableParallel<history>] s] Exiting Chain run with output:
[outputs]
[chain/end] [chai